In [1]:
from pathlib import Path
import re

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor

d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Resolve paths whether the notebook kernel starts in the project root or in src/.
BASE_DIR = Path.cwd().parent / "L30_data"

KEYFRAME_DIR = BASE_DIR / "keyframes"
EMBEDDING_DIR = BASE_DIR / "keyframe_embeddings"
MODEL_NAME = "google/siglip-so400m-patch14-384"
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

In [3]:
def natural_sort_key(path: Path):
    """Sort frame names numerically, e.g. 2.jpg before 10.jpg."""
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]


def list_video_dirs():
    if not KEYFRAME_DIR.is_dir():
        raise FileNotFoundError(f"Keyframe directory not found: {KEYFRAME_DIR}")
    return sorted(path for path in KEYFRAME_DIR.iterdir() if path.is_dir())


def load_images(image_paths):
    images = []
    for image_path in image_paths:
        try:
            with Image.open(image_path) as image:
                images.append(image.convert("RGB"))
        except (OSError, ValueError) as error:
            print(f"Skipping unreadable image {image_path}: {error}")
    return images


def extract_image_embeddings(model_output):
    """Handle both tensor outputs and model output objects from Transformers."""
    if isinstance(model_output, torch.Tensor):
        return model_output
    if hasattr(model_output, "image_embeds") and model_output.image_embeds is not None:
        return model_output.image_embeds
    if hasattr(model_output, "pooler_output") and model_output.pooler_output is not None:
        return model_output.pooler_output
    raise TypeError(f"Unsupported image output type: {type(model_output)}")


def embed_video(video_dir, processor, model):
    image_paths = sorted(
        (path for path in video_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS),
        key=natural_sort_key,
    )
    if not image_paths:
        print(f"Skipping {video_dir.name}: no image files found")
        return None

    video_embeddings = []
    for start in range(0, len(image_paths), BATCH_SIZE):
        images = load_images(image_paths[start:start + BATCH_SIZE])
        if not images:
            continue

        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)
        with torch.inference_mode():
            if hasattr(model, "get_image_features"):
                output = model.get_image_features(pixel_values=pixel_values)
            else:
                output = model(pixel_values=pixel_values)
            embeddings = extract_image_embeddings(output)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
        video_embeddings.append(embeddings.cpu().numpy().astype(np.float32))

    if not video_embeddings:
        return None
    return np.concatenate(video_embeddings, axis=0)


print(f"Repository: {BASE_DIR}")
print(f"Keyframes: {KEYFRAME_DIR}")
print(f"Output: {EMBEDDING_DIR}")
print(f"Device: {DEVICE}")

Repository: d:\Truong_pipeline\My_Code\L30_data
Keyframes: d:\Truong_pipeline\My_Code\L30_data\keyframes
Output: d:\Truong_pipeline\My_Code\L30_data\keyframe_embeddings
Device: cuda


In [4]:
# Load SigLIP and create one .npy file per video.
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading {MODEL_NAME} on {DEVICE}...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

Loading google/siglip-so400m-patch14-384 on cuda...


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 888/888 [00:00<00:00, 4535.84it/s]


SiglipModel(
  (text_model): SiglipTextModel(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 1152)
      (position_embedding): Embedding(64, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304, out_f

In [5]:
video_dirs = list_video_dirs()
print(f"Found {len(video_dirs)} video folders")

for video_dir in tqdm(video_dirs, desc="Embedding videos"):
    output_path = EMBEDDING_DIR / f"{video_dir.name}.npy"
    if output_path.exists():
        print(f"Skipping existing file: {output_path.name}")
        continue

    embeddings = embed_video(video_dir, processor, model)
    if embeddings is None:
        continue

    np.save(output_path, embeddings)
    print(f"Saved {output_path.name}: shape={embeddings.shape}")

print("Done.")

Found 96 video folders


Embedding videos:   1%|          | 1/96 [00:14<22:16, 14.07s/it]

Saved L30_V001.npy: shape=(72, 1152)


Embedding videos:   2%|▏         | 2/96 [00:28<22:44, 14.51s/it]

Saved L30_V002.npy: shape=(87, 1152)


...

Saved L30_V094.npy: shape=(82, 1152)


Embedding videos:  99%|█████████▉| 95/96 [23:52<00:18, 18.23s/it]

Saved L30_V095.npy: shape=(123, 1152)


Embedding videos: 100%|██████████| 96/96 [24:07<00:00, 15.08s/it]

Saved L30_V096.npy: shape=(84, 1152)
Done.
